# Day 022 | Pandas — Merge, Join & Concat

## 100 Days of ML for Geotechnical Engineering
**Phase:** Data Science Essentials

**Author:** Ripon Chandra Malo  
**Date:** YYYY-MM-DD  

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| `pd.merge()` | SQL-style join on key columns | Match lab results to field SPT by borehole + depth |
| Inner join | Keep only matching rows | Only depths where BOTH field and lab data exist |
| Left join | Keep all rows from left table | All field data, attach lab results where available |
| Outer join | Keep all rows from both tables | Complete picture — no data lost |
| `pd.concat()` | Stack DataFrames vertically or horizontally | Combine logs from multiple boreholes |
| `.join()` | Index-based merge shorthand | Quick merge when index = borehole_id |

---

## Our Geotechnical Problem

A site investigation produced **two separate datasets**:

1. **Field SPT data** — 20 rows from 4 boreholes with N-values and soil descriptions  
2. **Lab test results** — 15 rows with Atterberg limits, water content, and specific gravity  

Not every field sample was sent to the lab, and the lab tested some depths not logged in the field sheet. The engineer needs to **combine these into one comprehensive database**.

This is the most common data-wrangling task in geotechnics: matching field and lab records by borehole ID and depth.

---

## Part 1: Loading the Two Datasets

We have two CSV files from the same site investigation project. Let us load them and examine what keys they share.

In [ ]:
# ─────────────────────────────────────────────────────
# LOADING FIELD AND LAB DATASETS
# ─────────────────────────────────────────────────────

import pandas as pd
import numpy as np

# Load field SPT data (collected in the field during drilling)
field = pd.read_csv('field_spt_data.csv')
print("FIELD SPT DATA")
print(f"Shape: {field.shape[0]} rows x {field.shape[1]} columns")
print(f"Columns: {field.columns.tolist()}")
print(f"Boreholes: {field['borehole_id'].unique().tolist()}")
print()
print(field.head(10))
print()

# Load lab test results (from soil testing laboratory)
lab = pd.read_csv('lab_results.csv')
print("LAB TEST RESULTS")
print(f"Shape: {lab.shape[0]} rows x {lab.shape[1]} columns")
print(f"Columns: {lab.columns.tolist()}")
print(f"Boreholes: {lab['borehole_id'].unique().tolist()}")
print()
print(lab.head(10))

In [ ]:
# ─────────────────────────────────────────────────────
# IDENTIFYING COMMON KEYS
# ─────────────────────────────────────────────────────

# What columns do both tables share?
common_cols = set(field.columns) & set(lab.columns)
print(f"Common columns: {common_cols}")
print()

# How many unique (borehole, depth) pairs in each?
field_keys = set(zip(field['borehole_id'], field['depth_m']))
lab_keys = set(zip(lab['borehole_id'], lab['depth_m']))

print(f"Field records: {len(field_keys)} unique (borehole, depth) pairs")
print(f"Lab records:   {len(lab_keys)} unique (borehole, depth) pairs")
print(f"In BOTH:       {len(field_keys & lab_keys)} matching pairs")
print(f"Field only:    {len(field_keys - lab_keys)} (no lab data)")
print(f"Lab only:      {len(lab_keys - field_keys)} (no field data)")
print()
print("This mismatch is typical — not every sample goes to the lab,")
print("and the lab may test depths not in the SPT log.")

---

## Part 2: `pd.merge()` — Inner Join

An **inner join** keeps only rows where the key exists in **both** tables. This gives you complete records — every row has both field AND lab data.

```python
pd.merge(field, lab, on=['borehole_id', 'depth_m'], how='inner')
#         ↑ left    ↑ right   ↑ match on these keys   ↑ join type
```

Use inner join when you need **only complete records** for analysis.

In [ ]:
# ─────────────────────────────────────────────────────
# INNER JOIN — Only matching records
# ─────────────────────────────────────────────────────

inner = pd.merge(field, lab, on=['borehole_id', 'depth_m'], how='inner')

print(f"Inner join: {len(inner)} rows (only where both field & lab exist)")
print(f"Columns: {inner.columns.tolist()}")
print()
print(inner.to_string(index=False))
print()
print(f"We started with {len(field)} field rows and {len(lab)} lab rows.")
print(f"Inner join kept only {len(inner)} rows with COMPLETE data.")
print("Lost rows had either no lab test or no SPT record at that depth.")

---

## Part 3: `pd.merge()` — Left Join

A **left join** keeps ALL rows from the left table (field data), and attaches lab results where available. Missing lab values become `NaN`.

This is the most common join in geotechnics — you want to keep your complete field log and enrich it with lab data.

In [ ]:
# ─────────────────────────────────────────────────────
# LEFT JOIN — Keep all field data, attach lab where available
# ─────────────────────────────────────────────────────

left = pd.merge(field, lab, on=['borehole_id', 'depth_m'], how='left')

print(f"Left join: {len(left)} rows (all field records preserved)")
print()
print(left.to_string(index=False))
print()

# Count how many have lab data vs not
has_lab = left['liquid_limit'].notna().sum()
no_lab = left['liquid_limit'].isna().sum()
print(f"Rows with lab data:    {has_lab}")
print(f"Rows without lab data: {no_lab} (NaN in lab columns)")
print()
print("Left join is ideal for the field engineer's master log.")
print("Every SPT record stays; lab data fills in where tested.")

---

## Part 4: `pd.merge()` — Outer Join

An **outer join** keeps ALL rows from BOTH tables. No data is lost — you see every field record and every lab record, with `NaN` wherever one side has no match.

Use this when building a complete site database where you cannot afford to lose any information.

In [ ]:
# ─────────────────────────────────────────────────────
# OUTER JOIN — Keep everything from both tables
# ─────────────────────────────────────────────────────

outer = pd.merge(field, lab, on=['borehole_id', 'depth_m'], how='outer')
outer = outer.sort_values(['borehole_id', 'depth_m']).reset_index(drop=True)

print(f"Outer join: {len(outer)} rows (NOTHING lost from either table)")
print()
print(outer.to_string(index=False))
print()

# Identify rows that came from only one source
field_only = outer['N_value'].notna() & outer['liquid_limit'].isna()
lab_only = outer['N_value'].isna() & outer['liquid_limit'].notna()
both = outer['N_value'].notna() & outer['liquid_limit'].notna()

print(f"Both field & lab: {both.sum()} rows")
print(f"Field only:       {field_only.sum()} rows")
print(f"Lab only:         {lab_only.sum()} rows")
print()
print("Outer join gives the complete picture. Use indicator=True to")
print("see which source each row came from.")

In [ ]:
# ─────────────────────────────────────────────────────
# MERGE INDICATOR — Track data source
# ─────────────────────────────────────────────────────

# indicator=True adds a '_merge' column showing the source
outer_ind = pd.merge(field, lab, on=['borehole_id', 'depth_m'],
                     how='outer', indicator=True)
outer_ind = outer_ind.sort_values(['borehole_id', 'depth_m']).reset_index(drop=True)

print("Merge indicator shows data origin:")
print(outer_ind[['borehole_id', 'depth_m', '_merge']].to_string(index=False))
print()
print("Value counts:")
print(outer_ind['_merge'].value_counts().to_string())
print()
print("'both' = matched in field AND lab")
print("'left_only' = field data with no lab test")
print("'right_only' = lab data with no field SPT record")

---

## Part 5: `pd.concat()` — Stacking DataFrames

`pd.concat()` stacks DataFrames **vertically** (row-wise) or **horizontally** (column-wise). This is useful when you have separate DataFrames for each borehole and need to combine them into one.

```python
pd.concat([df1, df2, df3], axis=0)   # stack rows (vertical)
pd.concat([df1, df2], axis=1)         # stack columns (horizontal)
```

In [ ]:
# ─────────────────────────────────────────────────────
# CONCAT — Combining borehole DataFrames vertically
# ─────────────────────────────────────────────────────

# Simulate receiving data as separate DataFrames per borehole
bh01 = field[field['borehole_id'] == 'BH-01'].copy()
bh02 = field[field['borehole_id'] == 'BH-02'].copy()
bh03 = field[field['borehole_id'] == 'BH-03'].copy()
bh04 = field[field['borehole_id'] == 'BH-04'].copy()

print(f"BH-01: {len(bh01)} rows")
print(f"BH-02: {len(bh02)} rows")
print(f"BH-03: {len(bh03)} rows")
print(f"BH-04: {len(bh04)} rows")
print()

# Concat vertically (stack on top of each other)
combined = pd.concat([bh01, bh02, bh03, bh04], axis=0, ignore_index=True)

print(f"Combined: {len(combined)} rows")
print(combined.to_string())
print()
print("ignore_index=True resets the index to 0, 1, 2, ...")
print("Without it, the original indices from each DataFrame are kept.")

In [ ]:
# ─────────────────────────────────────────────────────
# CONCAT WITH KEYS — Track the source
# ─────────────────────────────────────────────────────

# Use keys to create a MultiIndex showing which DataFrame each row came from
combined_keys = pd.concat(
    [bh01, bh02, bh03],
    keys=['Borehole_1', 'Borehole_2', 'Borehole_3']
)

print("Concat with keys (MultiIndex):")
print(combined_keys[['depth_m', 'N_value', 'soil_description']].to_string())
print()

# Access a specific borehole from the MultiIndex
print("Accessing Borehole_2 from MultiIndex:")
print(combined_keys.loc['Borehole_2'][['depth_m', 'N_value']].to_string())

In [ ]:
# ─────────────────────────────────────────────────────
# CONCAT COLUMN-WISE — Side by side
# ─────────────────────────────────────────────────────

# Create summary stats for each borehole as separate Series
stats_bh01 = bh01['N_value'].describe().rename('BH-01')
stats_bh02 = bh02['N_value'].describe().rename('BH-02')
stats_bh03 = bh03['N_value'].describe().rename('BH-03')

# Concat horizontally to build a comparison table
comparison = pd.concat([stats_bh01, stats_bh02, stats_bh03], axis=1)

print("N-value statistics — side-by-side comparison:")
print(comparison.round(1).to_string())
print()
print("axis=1 places each Series as a column.")
print("This is how you build comparison tables for reports.")

---

## Part 6: `.join()` — Index-Based Merge

`.join()` is a shortcut for merging on the **index**. It is faster to type when your merge key is already the index.

```python
df1.set_index('borehole_id').join(df2.set_index('borehole_id'), lsuffix='_field', rsuffix='_lab')
```

In [ ]:
# ─────────────────────────────────────────────────────
# .join() — Index-based merge
# ─────────────────────────────────────────────────────

# Create summary DataFrames with borehole_id as index
field_summary = field.groupby('borehole_id').agg(
    avg_N=('N_value', 'mean'),
    max_N=('N_value', 'max'),
    n_layers=('N_value', 'count')
).round(1)

lab_summary = lab.groupby('borehole_id').agg(
    avg_LL=('liquid_limit', 'mean'),
    avg_wc=('water_content', 'mean'),
    avg_Gs=('specific_gravity', 'mean')
).round(2)

print("Field summary (index = borehole_id):")
print(field_summary.to_string())
print()
print("Lab summary (index = borehole_id):")
print(lab_summary.to_string())
print()

# .join() uses index by default
joined = field_summary.join(lab_summary)
print("Joined on index:")
print(joined.to_string())
print()
print(".join() is concise when both DataFrames share the same index.")

---

## Part 7: Handling Merge Pitfalls

Merging can go wrong in subtle ways. Here are the common issues in geotechnical data:

1. **Duplicate keys** — same borehole+depth appears twice  
2. **Key mismatches** — `BH-01` vs `BH01` vs `Borehole 1`  
3. **Column name conflicts** — both tables have a column called `depth`  
4. **Data type mismatches** — depth as `float` in one table, `int` in another

In [ ]:
# ─────────────────────────────────────────────────────
# HANDLING MERGE PITFALLS
# ─────────────────────────────────────────────────────

# Pitfall 1: Column name conflicts with suffixes
# If both tables had a 'depth' column (different names for the same thing)
field_v2 = field.rename(columns={'depth_m': 'depth'})
lab_v2 = lab.rename(columns={'depth_m': 'depth'})

# suffixes handle the conflict
merged_suffix = pd.merge(field_v2, lab_v2, on=['borehole_id', 'depth'],
                         how='inner')
print("Merge with shared column name (no conflict since we merge on it):")
print(merged_suffix.head(3).to_string(index=False))
print()

# Pitfall 2: Key format mismatches
lab_bad = lab.copy()
lab_bad['borehole_id'] = lab_bad['borehole_id'].str.replace('-', '')  # BH01 not BH-01

bad_merge = pd.merge(field, lab_bad, on=['borehole_id', 'depth_m'], how='inner')
print(f"Bad merge (key mismatch): {len(bad_merge)} rows — nothing matched!")
print("Always verify key format consistency before merging.")
print()

# Pitfall 3: validate parameter to catch duplicates
try:
    pd.merge(field, lab, on=['borehole_id', 'depth_m'],
             how='inner', validate='one_to_one')
    print("validate='one_to_one' passed — no duplicate keys.")
except Exception as e:
    print(f"Validation error: {e}")

---

## Part 8: Complete Application — Comprehensive Geotechnical Database

Let us combine everything to build a clean, merged geotechnical database from our field and lab data, compute derived parameters, and produce a summary report.

In [ ]:
# ══════════════════════════════════════════════════════
# COMPLETE: COMPREHENSIVE GEOTECHNICAL DATABASE
# ══════════════════════════════════════════════════════

print("╔" + "═" * 62 + "╗")
print("║  GEOTECHNICAL DATABASE — MERGE FIELD & LAB DATA             ║")
print("╚" + "═" * 62 + "╝")

# ── Step 1: Left join — keep all field data, attach lab results ──
geotech_db = pd.merge(field, lab, on=['borehole_id', 'depth_m'], how='left')

# ── Step 2: Derive plasticity index where Atterberg limits exist ──
geotech_db['plasticity_index'] = (geotech_db['liquid_limit'] -
                                   geotech_db['plastic_limit']).round(1)

# ── Step 3: Classify soil consistency from N-value ──
def classify_consistency(n):
    if n < 4: return 'Very Loose/Soft'
    elif n < 10: return 'Loose/Firm'
    elif n < 30: return 'Medium Dense/Stiff'
    elif n < 50: return 'Dense/Very Stiff'
    else: return 'Very Dense/Hard'

geotech_db['consistency'] = geotech_db['N_value'].apply(classify_consistency)

# ── Step 4: Flag data completeness ──
geotech_db['has_lab_data'] = geotech_db['liquid_limit'].notna()

print("\n1. MERGED DATABASE")
print("─" * 70)
print(geotech_db.to_string(index=False))

In [ ]:
# ══════════════════════════════════════════════════════
# COMPLETE (continued): Summary Report
# ══════════════════════════════════════════════════════

# ── Section 2: Data Completeness Report ──
print("2. DATA COMPLETENESS REPORT")
print("─" * 70)

completeness = geotech_db.groupby('borehole_id').agg(
    total_layers=('N_value', 'count'),
    layers_with_lab=('has_lab_data', 'sum'),
    lab_coverage=('has_lab_data', 'mean')
)
completeness['lab_coverage'] = (completeness['lab_coverage'] * 100).round(0).astype(int).astype(str) + '%'
print(completeness.to_string())
print()

# ── Section 3: Borehole Summary ──
print("\n3. BOREHOLE SUMMARY")
print("─" * 70)

bh_summary = geotech_db.groupby('borehole_id').agg(
    avg_N=('N_value', 'mean'),
    max_depth=('depth_m', 'max'),
    avg_wc=('water_content', 'mean'),
    avg_PI=('plasticity_index', 'mean')
).round(1)
print(bh_summary.to_string())
print()

# ── Section 4: Consistency Classification ──
print("\n4. CONSISTENCY DISTRIBUTION")
print("─" * 70)
consistency_dist = pd.crosstab(geotech_db['consistency'],
                                geotech_db['borehole_id'],
                                margins=True)
print(consistency_dist.to_string())
print()

print("\n5. ENGINEERING NOTES")
print("─" * 70)
total = len(geotech_db)
with_lab = geotech_db['has_lab_data'].sum()
print(f"  Total records:           {total}")
print(f"  Records with lab data:   {with_lab} ({with_lab/total*100:.0f}%)")
print(f"  Records without lab:     {total - with_lab} ({(total-with_lab)/total*100:.0f}%)")
print(f"  Site average N-value:    {geotech_db['N_value'].mean():.1f}")
print(f"  Deepest borehole:        {geotech_db['depth_m'].max()} m")
print("═" * 70)

---

## Practice Exercises

### Exercise 1: Right Join
Perform a **right join** to keep all lab records and attach field data where available. How many lab-only records exist (no matching field SPT)?

### Exercise 2: Concat with New Borehole
Create a new DataFrame for `BH-05` with 4 rows of SPT data. Use `pd.concat()` to add it to the field DataFrame. Verify the combined shape.

### Exercise 3: Multi-Source Merge
Create a third DataFrame with CPT data (borehole_id, depth_m, qc, fs) for BH-01 and BH-02. Merge all three sources (field, lab, CPT) into one master DataFrame using successive merges.

In [ ]:
# EXERCISE 1
# right = pd.merge(field, lab, on=['borehole_id', 'depth_m'], how='right')
# lab_only = right['N_value'].isna().sum()
# print(f"Lab-only records (no field SPT): {lab_only}")
# print(right.to_string(index=False))

In [ ]:
# EXERCISE 2
# bh05 = pd.DataFrame({
#     'borehole_id': ['BH-05'] * 4,
#     'depth_m': [1.5, 3.0, 4.5, 6.0],
#     'N_value': [6, 11, 19, 25],
#     'soil_description': ['Fill', 'Silty sand', 'Sand', 'Dense sand']
# })
# field_ext = pd.concat([field, bh05], ignore_index=True)
# print(f"Original: {len(field)} rows, Extended: {len(field_ext)} rows")

In [ ]:
# EXERCISE 3
# cpt = pd.DataFrame({
#     'borehole_id': ['BH-01','BH-01','BH-01','BH-02','BH-02','BH-02'],
#     'depth_m': [1.5, 3.0, 6.0, 1.5, 3.0, 6.0],
#     'qc_MPa': [1.2, 3.5, 8.1, 0.8, 2.1, 6.5],
#     'fs_kPa': [15, 45, 95, 10, 30, 78]
# })
# master = pd.merge(field, lab, on=['borehole_id', 'depth_m'], how='outer')
# master = pd.merge(master, cpt, on=['borehole_id', 'depth_m'], how='outer')
# print(master.to_string(index=False))

---

## Key Takeaways

1. **`pd.merge()`** is the workhorse for combining datasets on shared keys. In geotechnics, your keys are almost always `borehole_id` and `depth`.

2. **Inner join** gives complete records only. **Left join** preserves your primary dataset. **Outer join** loses nothing. Choose based on your analysis needs.

3. **`pd.concat()`** stacks DataFrames vertically (combining borehole logs) or horizontally (building comparison tables). Use `ignore_index=True` to reset row numbering.

4. **`.join()`** is a concise index-based merge — perfect when your DataFrames are already indexed by borehole_id.

5. Always check for **key mismatches** (`BH-01` vs `BH01`), **duplicate keys**, and **data type conflicts** before merging. Use `indicator=True` and `validate=` to catch problems early.

---

## How This Connects to ML

| Today's Concept | ML Equivalent |
|----------------|---------------|
| `pd.merge()` inner join | Joining train/test features with labels |
| Left join with NaN | Feature matrix with missing values (needs imputation) |
| Outer join | Building complete feature sets from multiple data sources |
| `pd.concat()` rows | Combining datasets from multiple sites for transfer learning |
| `pd.concat()` columns | Adding engineered features alongside original columns |
| `.join()` on index | Attaching predictions back to the original DataFrame |
| `validate=` parameter | Data integrity checks before model training |
| `indicator=True` | Tracking which data source each training sample came from |

In ML pipelines, you constantly merge features from different sources — field tests, lab results, geophysical surveys, weather data. Mastering merge and concat is essential for building rich feature matrices.

---

*Day 022 of 100 — Merging data is where real-world geotechnical datasets come together!*